In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D
from plot_style import *

params = {
    'R'  : 1.0,
    'L'  : 0.01,
    'Ke' : 0.01,
    'Kt' : 0.01,
    'J'  : 0.01,
    'B'  : 0.001,
    'V'  : 12.0,
    'TL' : 0.05,
}

STIMULUS_PROFILE = {
    'load_torque': {'time': 2.0, 'multiplier': 1.05},   
    'viscous_drag': {'time': 3.0, 'multiplier': 1.10}, 
    'voltage_dip': {'time': 4.0, 'multiplier': 0.97},  
}

def step_multiplier(t, profile, name):
    step = profile.get(name)
    return 1.0 if step is None or t < step['time'] else step['multiplier']

def motor_derivatives(t, state, params):
    i     = state[0]
    omega = state[1]

    R  = params['R']
    L  = params['L']
    Ke = params['Ke']
    Kt = params['Kt']
    J  = params['J']
    profile = params.get('stimulus_profile', STIMULUS_PROFILE)
    B  = params['B'] * step_multiplier(t, profile, 'viscous_drag')
    V  = params['V'] * step_multiplier(t, profile, 'voltage_dip')
    TL = params['TL'] * step_multiplier(t, profile, 'load_torque')

    di_dt     = (V - R*i - Ke*omega) / L
    domega_dt = (Kt*i - B*omega - TL) / J

    return [di_dt, domega_dt]

def steady_state_initial_state(sim_params):
    R, B = sim_params['R'], sim_params['B']
    Ke, Kt = sim_params['Ke'], sim_params['Kt']
    omega_0 = (Kt * sim_params['V'] / R - sim_params['TL']) / (B + Kt * Ke / R)
    current_0 = (sim_params['V'] - Ke * omega_0) / R
    return [current_0, omega_0]


t_span = (0, 5)
t_eval = np.linspace(0, 5, 5000)
initial_state = steady_state_initial_state(params)

solution = solve_ivp(
    fun=motor_derivatives,
    t_span=t_span,
    y0=initial_state,
    t_eval=t_eval,
    args=(params,),
    method='RK45',
    rtol=1e-6,
    atol=1e-8
)

time    = solution.t
current = solution.y[0]
speed   = solution.y[1]

#plot
apply_base_style()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6.5), gridspec_kw={'hspace': 0.08})

fig.text(0.055, 0.97, 'DC motor — controlled multi-parameter test',
         fontsize=13, fontweight='500', color=PALETTE['title'], va='top', ha='left')
fig.text(0.055, 0.925, 'Armature current and angular speed under sequential diagnostic stimuli',
         fontsize=9, color=PALETTE['label'], va='top', ha='left')

ax1.plot(time, current, color=PALETTE['blue'], linewidth=2.0, zorder=3)
ax1.fill_between(time, current, current.min(), alpha=0.07, color=PALETTE['blue'], zorder=2)
draw_event_lines(ax1)
ax1.set_ylabel('Current (A)')
ax1.set_xlim(0, 5)
ax1.tick_params(labelbottom=False)
ax1.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))

idx_dip = np.searchsorted(time, 4.0)
ax1.annotate(
    f'−{current[idx_dip-1] - current[-1]:.3f} A at voltage dip',
    xy=(4.0, current[idx_dip]),
    xytext=(4.25, current[idx_dip] + 0.08),
    fontsize=7.5, color=PALETTE['event_volt'],
    arrowprops=dict(arrowstyle='->', color=PALETTE['event_volt'],
                    lw=0.9, connectionstyle='arc3,rad=0.2'),
)
add_event_legend(ax1)

ax2.plot(time, speed, color=PALETTE['orange'], linewidth=2.0, zorder=3)
ax2.fill_between(time, speed, speed.min(), alpha=0.07, color=PALETTE['orange'], zorder=2)
draw_event_lines(ax2)
ax2.set_ylabel('Speed (rad/s)')
ax2.set_xlabel('Time (s)')
ax2.set_xlim(0, 5)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(1))
ax2.annotate(
    f'−{speed[0] - speed[-1]:.2f} rad/s total',
    xy=(5.0, speed[-1]),
    xytext=(3.6, speed[-1] - 0.45),
    fontsize=7.5, color=PALETTE['orange'],
    arrowprops=dict(arrowstyle='->', color=PALETTE['orange'],
                    lw=0.9, connectionstyle='arc3,rad=-0.2'),
)

plt.savefig('../plots/healthy_motor_response.png')
plt.close()
print("Simulation complete. Plot saved to plots folder.")

findfont: Failed to find font weight 500, now using 400.


Simulation complete. Plot saved to plots folder.


In [ ]:

def run_motor_simulation(sim_params, duration=5.0, n_samples=5000):
    
    result = solve_ivp(
        fun=motor_derivatives,
        t_span=(0, duration),
        y0=steady_state_initial_state(sim_params),
        t_eval=np.linspace(0, duration, n_samples),
        args=(sim_params,),
        method='RK45',
        rtol=1e-6,
        atol=1e-8
    )
    if not result.success:
        raise RuntimeError(f'Motor simulation failed: {result.message}')
    return result.t, result.y[0], result.y[1]

fault_parameters = {
    'Healthy': params.copy(),
    'Bearing fault (B x2.0)': {**params, 'B': params['B'] * 2.0},
    'Winding fault (R x1.5)': {**params, 'R': params['R'] * 1.5},
    'Mechanical fault (J x1.3)': {**params, 'J': params['J'] * 1.3},
    'Supply fault (V x0.90)': {**params, 'V': params['V'] * 0.90},
}

responses = {
    label: run_motor_simulation(sim_params)
    for label, sim_params in fault_parameters.items()
}

apply_base_style()

fig, (ax_current, ax_speed) = plt.subplots(2, 1, figsize=(11, 7.5), gridspec_kw={'hspace': 0.08})

fig.text(0.055, 0.97, 'Healthy vs degraded DC motor — fault comparison',
         fontsize=13, fontweight='500', color=PALETTE['title'], va='top', ha='left')
fig.text(0.055, 0.93, 'Five motor conditions under identical diagnostic stimulus sequence',
         fontsize=9, color=PALETTE['label'], va='top', ha='left')

for label, (sim_time, sim_current, sim_speed) in responses.items():
    color = FAULT_COLORS[label]
    dash  = FAULT_DASHES[label]
    lw    = 2.2 if label == 'Healthy' else 1.6
    ls    = (0, dash) if dash[0] else 'solid'
    kw    = dict(color=color, linewidth=lw, linestyle=ls,
                 zorder=4 if label == 'Healthy' else 3)
    ax_current.plot(sim_time, sim_current, **kw)
    ax_speed.plot(sim_time, sim_speed, **kw)

for ax in (ax_current, ax_speed):
    draw_event_lines(ax)
    ax.set_xlim(0, 5)

ax_current.tick_params(labelbottom=False)
ax_current.set_ylabel('Current (A)')
ax_speed.set_ylabel('Speed (rad/s)')
ax_speed.set_xlabel('Time (s)')
ax_speed.xaxis.set_major_locator(ticker.MultipleLocator(1))

series_handles = [
    Line2D([0], [0],
           color=FAULT_COLORS[lbl],
           linewidth=2.2 if lbl == 'Healthy' else 1.6,
           linestyle=(0, FAULT_DASHES[lbl]) if FAULT_DASHES[lbl][0] else 'solid',
           label=lbl)
    for lbl in fault_parameters
]
leg = ax_current.legend(
    handles=series_handles, loc='upper right', fontsize=8,
    frameon=True, framealpha=0.92, edgecolor=PALETTE['border'],
    facecolor=PALETTE['panel'], handlelength=2.4,
    borderpad=0.7, labelspacing=0.45,
    title='Motor condition', title_fontsize=8,
)
leg.get_title().set_color(PALETTE['label'])

event_handles = [
    Line2D([0], [0], color=m['color'], linewidth=1.2,
           linestyle='--', alpha=0.85, label=m['label'])
    for m in EVENT_META.values()
]
leg2 = ax_speed.legend(
    handles=event_handles, loc='lower left', fontsize=8,
    frameon=True, framealpha=0.92, edgecolor=PALETTE['border'],
    facecolor=PALETTE['panel'], handlelength=2.2,
    borderpad=0.7, labelspacing=0.45,
    title='Stimulus events', title_fontsize=8,
)
leg2.get_title().set_color(PALETTE['label'])

plt.savefig('../plots/healthy_vs_degraded_responses.png')
plt.close()
print('Stage 2 complete. Saved plots/healthy_vs_degraded_responses.png')


Stage 2 complete. Saved plots/healthy_vs_degraded_responses.png


In [ ]:

from scipy.stats import kurtosis

FEATURE_NAMES = [
    'rpm_drop_pct', 'settling_time_s', 'current_peak_a', 'current_rms_a',
    'current_rise_pct', 'speed_rms_rad_s', 'overshoot_pct',
    'speed_kurtosis', 'current_kurtosis', 'current_crest_factor'
]

def extract_fingerprint(time, current, speed, stimulus_time=2.0):
    
    time = np.asarray(time)
    current = np.asarray(current)
    speed = np.asarray(speed)
    post_mask = time >= stimulus_time
    if post_mask.sum() < 10:
        raise ValueError('At least ten post-stimulus samples are required.')

    t = time[post_mask]
    i = current[post_mask]
    w = speed[post_mask]
    i_initial, w_initial = i[0], w[0]
    tail_start = max(1, int(0.9 * len(t)))
    i_final = np.mean(i[tail_start:])
    w_final = np.mean(w[tail_start:])

    #positive rpm drop
    rpm_drop_pct = 100 * max(0.0, (w_initial - np.min(w)) / max(abs(w_initial), 1e-12))
    current_peak_a = np.max(np.abs(i))
    current_rms_a = np.sqrt(np.mean(i ** 2))
    current_rise_pct = 100 * max(0.0, (np.max(i) - i_initial) / max(abs(i_initial), 1e-12))
    speed_rms_rad_s = np.sqrt(np.mean(w ** 2))

    #overshoot measure
    speed_change = w_final - w_initial
    if abs(speed_change) < 1e-12:
        overshoot_pct = 0.0
        settling_time_s = 0.0
    else:
        direction = np.sign(speed_change)
        overshoot_pct = 100 * max(0.0, np.max(direction * (w - w_final)) / abs(speed_change))
        tolerance = 0.02 * abs(speed_change)
        outside_band = np.abs(w - w_final) > tolerance
        settling_index = np.where(outside_band)[0][-1] + 1 if np.any(outside_band) else 0
        settling_time_s = (t[min(settling_index, len(t) - 1)] - stimulus_time)

    fingerprint = {
        'rpm_drop_pct': rpm_drop_pct,
        'settling_time_s': settling_time_s,
        'current_peak_a': current_peak_a,
        'current_rms_a': current_rms_a,
        'current_rise_pct': current_rise_pct,
        'speed_rms_rad_s': speed_rms_rad_s,
        'overshoot_pct': overshoot_pct,
        'speed_kurtosis': kurtosis(w, fisher=False, bias=False),
        'current_kurtosis': kurtosis(i, fisher=False, bias=False),
        'current_crest_factor': current_peak_a / max(current_rms_a, 1e-12),
    }
    return fingerprint

fingerprints = {
    label: extract_fingerprint(sim_time, sim_current, sim_speed)
    for label, (sim_time, sim_current, sim_speed) in responses.items()
}
for label, fingerprint in fingerprints.items():
    print(f'\n{label}')
    for feature in FEATURE_NAMES:
        print(f'  {feature}: {fingerprint[feature]:.5f}')



Healthy
  rpm_drop_pct: 3.29488
  settling_time_s: 3.00000
  current_peak_a: 11.37401
  current_rms_a: 11.25376
  current_rise_pct: 0.09126
  speed_rms_rad_s: 62.85705
  overshoot_pct: 7.73587
  speed_kurtosis: 1.91270
  current_kurtosis: 1.52422
  current_crest_factor: 1.01068

Bearing fault (B x2.0)
  rpm_drop_pct: 5.79682
  settling_time_s: 3.00000
  current_peak_a: 11.67659
  current_rms_a: 11.55633
  current_rise_pct: 0.08515
  speed_rms_rad_s: 32.59625
  overshoot_pct: 7.05665
  speed_kurtosis: 1.84612
  current_kurtosis: 1.52325
  current_crest_factor: 1.01041

Winding fault (R x1.5)
  rpm_drop_pct: 4.83813
  settling_time_s: 3.00000
  current_peak_a: 7.81724
  current_rms_a: 7.73743
  current_rise_pct: 0.06073
  speed_rms_rad_s: 27.57890
  overshoot_pct: 7.30622
  speed_kurtosis: 1.90758
  current_kurtosis: 1.51522
  current_crest_factor: 1.01031

Mechanical fault (J x1.3)
  rpm_drop_pct: 2.61432
  settling_time_s: 3.00000
  current_peak_a: 11.37182
  current_rms_a: 11.25212
 

In [ ]:

DATASET_CLASS_SPECS = {
    'Healthy': {
        'fault_multiplier_ranges': {},
        'description': 'Normal motor with small manufacturing/operating variation.',
    },
    'Bearing fault': {
        'fault_multiplier_ranges': {'B': (1.30, 2.50)},
        'description': 'Higher permanent viscous friction from bearing wear.',
    },
    'Winding fault': {
        'fault_multiplier_ranges': {'R': (1.20, 2.00)},
        'description': 'Higher permanent winding resistance from damage or poor connection.',
    },
    'Mechanical fault': {
        'fault_multiplier_ranges': {'J': (1.10, 1.40)},
        'description': 'Higher permanent inertia from imbalance or added rotating mass.',
    },
    'Supply fault': {
        'fault_multiplier_ranges': {'V': (0.85, 0.95)},
        'description': 'Persistently reduced available supply voltage.',
    },
}


HEALTHY_VARIATION_RANGE = (0.95, 1.05)
NOISE_FRACTION_OF_STD = 0.02
# Mild-fault ranges for the stress set. Each range starts where the training range ends,
# so the model has never seen these severities. Used for TESTING only, never for training.
STRESS_CLASS_SPECS = {
    'Healthy':          {'fault_multiplier_ranges': {}},
    'Bearing fault':    {'fault_multiplier_ranges': {'B': (1.05, 1.30)}},
    'Winding fault':    {'fault_multiplier_ranges': {'R': (1.05, 1.20)}},
    'Mechanical fault': {'fault_multiplier_ranges': {'J': (1.03, 1.10)}},
    'Supply fault':     {'fault_multiplier_ranges': {'V': (0.95, 0.98)}},
}

print('Dataset setup ready. Classes:', ', '.join(DATASET_CLASS_SPECS))
print('Shared diagnostic sequence:', STIMULUS_PROFILE)


Dataset setup ready. Classes: Healthy, Bearing fault, Winding fault, Mechanical fault, Supply fault
Shared diagnostic sequence: {'load_torque': {'time': 2.0, 'multiplier': 1.05}, 'viscous_drag': {'time': 3.0, 'multiplier': 1.1}, 'voltage_dip': {'time': 4.0, 'multiplier': 0.97}}


In [ ]:

import pandas as pd
from pathlib import Path

RANDOM_SEED = 20260918
SAMPLES_PER_CLASS = 12_000
DATASET_SIMULATION_SAMPLES = 1_001  


NORMAL_PARAMETER_RANGES = {
    'R': (0.97, 1.03),
    'L': (0.95, 1.05),
    'J': (0.95, 1.05),
    'B': (0.90, 1.10),
    'V': (0.99, 1.01),
    'TL': (0.95, 1.05),
}
GAIN_RANGE = (0.98, 1.02)  

def sample_realistic_parameters(label, rng, class_specs=None):
    """Sample one plausible motor, then apply only the requested permanent fault."""
    class_specs = class_specs or DATASET_CLASS_SPECS
    sampled = params.copy()
    for name, bounds in NORMAL_PARAMETER_RANGES.items():
        sampled[name] *= rng.uniform(*bounds)

    gain_factor = rng.uniform(*GAIN_RANGE)
    sampled['Ke'] *= gain_factor
    sampled['Kt'] *= gain_factor

    fault_multipliers = {}
    for name, bounds in class_specs[label]['fault_multiplier_ranges'].items():
        multiplier = rng.uniform(*bounds)
        sampled[name] *= multiplier
        fault_multipliers[name] = multiplier
    return sampled, fault_multipliers

def simulate_motor_batch(parameter_arrays, duration=5.0, n_samples=DATASET_SIMULATION_SAMPLES):

    t = np.linspace(0.0, duration, n_samples)
    n = len(parameter_arrays['R'])
    R, L = parameter_arrays['R'].to_numpy(), parameter_arrays['L'].to_numpy()
    Ke, Kt = parameter_arrays['Ke'].to_numpy(), parameter_arrays['Kt'].to_numpy()
    J, B = parameter_arrays['J'].to_numpy(), parameter_arrays['B'].to_numpy()
    V, TL = parameter_arrays['V'].to_numpy(), parameter_arrays['TL'].to_numpy()

    omega = (Kt * V / R - TL) / (B + Kt * Ke / R)
    current = (V - Ke * omega) / R
    current_history = np.empty((n, n_samples))
    speed_history = np.empty((n, n_samples))
    current_history[:, 0], speed_history[:, 0] = current, omega

    def derivatives(i, w, time_point):
        b_eff = B * step_multiplier(time_point, STIMULUS_PROFILE, 'viscous_drag')
        v_eff = V * step_multiplier(time_point, STIMULUS_PROFILE, 'voltage_dip')
        tl_eff = TL * step_multiplier(time_point, STIMULUS_PROFILE, 'load_torque')
        return (v_eff - R * i - Ke * w) / L, (Kt * i - b_eff * w - tl_eff) / J

    for index in range(n_samples - 1):
        dt = t[index + 1] - t[index]
        k1_i, k1_w = derivatives(current, omega, t[index])
        k2_i, k2_w = derivatives(current + 0.5 * dt * k1_i, omega + 0.5 * dt * k1_w, t[index] + 0.5 * dt)
        k3_i, k3_w = derivatives(current + 0.5 * dt * k2_i, omega + 0.5 * dt * k2_w, t[index] + 0.5 * dt)
        k4_i, k4_w = derivatives(current + dt * k3_i, omega + dt * k3_w, t[index + 1])
        current += dt * (k1_i + 2 * k2_i + 2 * k3_i + k4_i) / 6
        omega += dt * (k1_w + 2 * k2_w + 2 * k3_w + k4_w) / 6
        current_history[:, index + 1], speed_history[:, index + 1] = current, omega
    return t, current_history, speed_history

def extract_fingerprints_batch(time, current, speed):
    post = time >= 2.0
    t, i, w = time[post], current[:, post], speed[:, post]
    i_initial, w_initial = i[:, 0], w[:, 0]
    tail_start = max(1, int(0.9 * len(t)))
    w_final = np.mean(w[:, tail_start:], axis=1)
    speed_change = w_final - w_initial
    safe_change = np.maximum(np.abs(speed_change), 1e-12)
    direction = np.sign(speed_change)
    outside_band = np.abs(w - w_final[:, None]) > (0.02 * safe_change)[:, None]
    has_outside = np.any(outside_band, axis=1)
    last_outside = len(t) - 1 - np.argmax(outside_band[:, ::-1], axis=1)
    settling_index = np.where(has_outside, np.minimum(last_outside + 1, len(t) - 1), 0)
    current_peak = np.max(np.abs(i), axis=1)
    current_rms = np.sqrt(np.mean(i ** 2, axis=1))
    return pd.DataFrame({
        'rpm_drop_pct': 100 * np.maximum(0.0, (w_initial - np.min(w, axis=1)) / np.maximum(np.abs(w_initial), 1e-12)),
        'settling_time_s': t[settling_index] - 2.0,
        'current_peak_a': current_peak,
        'current_rms_a': current_rms,
        'current_rise_pct': 100 * np.maximum(0.0, (np.max(i, axis=1) - i_initial) / np.maximum(np.abs(i_initial), 1e-12)),
        'speed_rms_rad_s': np.sqrt(np.mean(w ** 2, axis=1)),
        'overshoot_pct': 100 * np.maximum(0.0, np.max(direction[:, None] * (w - w_final[:, None]), axis=1) / safe_change),
        'speed_kurtosis': kurtosis(w, axis=1, fisher=False, bias=False),
        'current_kurtosis': kurtosis(i, axis=1, fisher=False, bias=False),
        'current_crest_factor': current_peak / np.maximum(current_rms, 1e-12),
    })

def generate_fingerprint_dataset(samples_per_class=SAMPLES_PER_CLASS, seed=RANDOM_SEED, class_specs=None):
    class_specs = class_specs or DATASET_CLASS_SPECS
    rng = np.random.default_rng(seed)
    feature_frames, audit_frames = [], []
    sample_id = 0
    total_samples = samples_per_class * len(class_specs)

    for label in class_specs:
        sampled_records, audit_records = [], []
        for offset in range(samples_per_class):
            sampled_params, fault_multipliers = sample_realistic_parameters(label, rng, class_specs)
            multiplier = next(iter(fault_multipliers.values()), 1.0)   # 1.0 = healthy
            sampled_records.append(sampled_params)
            audit_records.append({
                'sample_id': sample_id + offset, 'label': label, **sampled_params,
                'fault_multipliers': str(fault_multipliers),
                'fault_multiplier': multiplier,
                'severity': abs(1.0 - multiplier),
            })

        parameter_frame = pd.DataFrame(sampled_records)
        sim_time, sim_current, sim_speed = simulate_motor_batch(parameter_frame)
        current_noise_fraction = rng.uniform(0.005, NOISE_FRACTION_OF_STD, size=samples_per_class)
        speed_noise_fraction = rng.uniform(0.005, NOISE_FRACTION_OF_STD, size=samples_per_class)
        sim_current += rng.normal(0, np.std(sim_current, axis=1)[:, None] * current_noise_fraction[:, None], sim_current.shape)
        sim_speed += rng.normal(0, np.std(sim_speed, axis=1)[:, None] * speed_noise_fraction[:, None], sim_speed.shape)

        class_features = extract_fingerprints_batch(sim_time, sim_current, sim_speed)
        if not np.all(np.isfinite(class_features[FEATURE_NAMES].to_numpy())):
            raise ValueError(f'Non-finite fingerprint in {label} samples.')
        class_features.insert(0, 'sample_id', np.arange(sample_id, sample_id + samples_per_class))
        class_features['label'] = label
        feature_frames.append(class_features)

        class_audit = pd.DataFrame(audit_records)
        class_audit['current_noise_fraction'] = current_noise_fraction
        class_audit['speed_noise_fraction'] = speed_noise_fraction
        audit_frames.append(class_audit)
        sample_id += samples_per_class
        print(f'Completed {label}: {sample_id:,}/{total_samples:,} samples')

    return pd.concat(feature_frames, ignore_index=True), pd.concat(audit_frames, ignore_index=True)

fingerprint_dataset, parameter_audit = generate_fingerprint_dataset()
output_dir = Path('../data')
output_dir.mkdir(parents=True, exist_ok=True)
fingerprint_dataset.to_csv(output_dir / 'fingerprint_dataset.csv', index=False)
parameter_audit.to_csv(output_dir / 'fingerprint_parameter_audit.csv', index=False)

# Row-aligned severity file for the severity breakdown (same row order as the normalized CSV)
fingerprint_dataset[['sample_id', 'label']].join(
    parameter_audit[['fault_multiplier', 'severity']]
).to_csv(output_dir / 'fingerprint_metadata.csv', index=False)

print(f'\nSaved {len(fingerprint_dataset):,} ML-ready rows to data/fingerprint_dataset.csv')
print('Saved source-parameter audit trail to data/fingerprint_parameter_audit.csv')
print(fingerprint_dataset['label'].value_counts().sort_index())


Completed Healthy: 12,000/60,000 samples
Completed Bearing fault: 24,000/60,000 samples
Completed Winding fault: 36,000/60,000 samples
Completed Mechanical fault: 48,000/60,000 samples
Completed Supply fault: 60,000/60,000 samples

Saved 60,000 ML-ready rows to data/fingerprint_dataset.csv
Saved source-parameter audit trail to data/fingerprint_parameter_audit.csv
label
Bearing fault       12000
Healthy             12000
Mechanical fault    12000
Supply fault        12000
Winding fault       12000
Name: count, dtype: int64


In [ ]:

import pandas as pd
raw_dataset = pd.read_csv('../data/fingerprint_dataset.csv')
parameter_audit = pd.read_csv('../data/fingerprint_parameter_audit.csv')

def propagate_linear_motor(i, w, R, L, Ke, Kt, J, B, V, TL, duration):
    a, b = -R / L, -Ke / L
    c, d = Kt / J, -B / J
    q_i, q_w = V / L, -TL / J
    determinant = a * d - b * c
    i_ss = (-d * q_i + b * q_w) / determinant
    w_ss = (c * q_i - a * q_w) / determinant
    trace_half = (a + d) / 2
    root = np.sqrt(((a - d) / 2) ** 2 + b * c)
    slow, fast = trace_half + root, trace_half - root
    denominator = slow - fast
    exp_slow, exp_fast = np.exp(slow * duration), np.exp(fast * duration)
    e11 = (exp_slow * (a - fast) - exp_fast * (a - slow)) / denominator
    e12 = b * (exp_slow - exp_fast) / denominator
    e21 = c * (exp_slow - exp_fast) / denominator
    e22 = (exp_slow * (d - fast) - exp_fast * (d - slow)) / denominator
    return i_ss + e11 * (i - i_ss) + e12 * (w - w_ss), w_ss + e21 * (i - i_ss) + e22 * (w - w_ss)

def calculate_extended_settling_time(audit, horizon_s=60.0, step_s=0.02):
    """Return 2%-band settling time measured from the first t=2 s input."""
    R, L = audit['R'].to_numpy(), audit['L'].to_numpy()
    Ke, Kt = audit['Ke'].to_numpy(), audit['Kt'].to_numpy()
    J, B = audit['J'].to_numpy(), audit['B'].to_numpy()
    V, TL = audit['V'].to_numpy(), audit['TL'].to_numpy()

    # State is at the steady operating point until the load step at t=2 s.
    i0 = (V - Ke * ((Kt * V / R - TL) / (B + Kt * Ke / R))) / R
    w0 = (Kt * V / R - TL) / (B + Kt * Ke / R)
    i3, w3 = propagate_linear_motor(i0, w0, R, L, Ke, Kt, J, B, V, TL * 1.05, 1.0)
    i4, w4 = propagate_linear_motor(i3, w3, R, L, Ke, Kt, J, B * 1.10, V, TL * 1.05, 1.0)

    # Final dynamics after the voltage dip; obtain its steady state and eigenmodes.
    final_v, final_b, final_tl = V * 0.97, B * 1.10, TL * 1.05
    a, b, c, d = -R / L, -Ke / L, Kt / J, -final_b / J
    determinant = a * d - b * c
    q_i, q_w = final_v / L, -final_tl / J
    w_final = (c * q_i - a * q_w) / determinant
    trace_half = (a + d) / 2
    root = np.sqrt(((a - d) / 2) ** 2 + b * c)
    slow, fast = trace_half + root, trace_half - root
    mode_denominator = slow - fast
    slow_coefficient = (c * (i4 - ( -d * q_i + b * q_w) / determinant) + (d - fast) * (w4 - w_final)) / mode_denominator
    fast_coefficient = (-c * (i4 - ( -d * q_i + b * q_w) / determinant) - (d - slow) * (w4 - w_final)) / mode_denominator
    tolerance = 0.02 * np.maximum(np.abs(w4 - w_final), 1e-9)
    timeline = np.arange(0.0, horizon_s + step_s, step_s)
    settling_after_final = np.empty(len(audit))

    for start in range(0, len(audit), 1_000):
        stop = min(start + 1_000, len(audit))
        error = np.abs(
            slow_coefficient[start:stop, None] * np.exp(slow[start:stop, None] * timeline)
            + fast_coefficient[start:stop, None] * np.exp(fast[start:stop, None] * timeline)
        )
        outside = error > tolerance[start:stop, None]
        last_outside = len(timeline) - 1 - np.argmax(outside[:, ::-1], axis=1)
        has_outside = np.any(outside, axis=1)
        settling_index = np.where(has_outside, np.minimum(last_outside + 1, len(timeline) - 1), 0)
        settling_after_final[start:stop] = timeline[settling_index]

    return 2.0 + settling_after_final

raw_dataset['settling_time_s'] = calculate_extended_settling_time(parameter_audit)
if raw_dataset['settling_time_s'].nunique() <= 1:
    raise ValueError('Extended settling calculation did not produce useful variation.')
raw_dataset.to_csv('../data/fingerprint_dataset.csv', index=False)

if raw_dataset.isna().any().any():
    raise ValueError('Dataset contains missing values.')
if raw_dataset['label'].value_counts().nunique() != 1:
    raise ValueError('Dataset classes are not balanced.')

constant_features = [feature for feature in FEATURE_NAMES if raw_dataset[feature].nunique() <= 1]
MODEL_FEATURE_NAMES = [feature for feature in FEATURE_NAMES if feature not in constant_features]
if not MODEL_FEATURE_NAMES:
    raise ValueError('No variable features remain for ML training.')


healthy_rows = raw_dataset.loc[raw_dataset['label'] == 'Healthy', MODEL_FEATURE_NAMES]
baseline_mean = healthy_rows.mean()
baseline_std = healthy_rows.std(ddof=0)
if (baseline_std <= 0).any():
    raise ValueError('A retained feature has zero healthy-baseline standard deviation.')

ml_ready_dataset = raw_dataset[MODEL_FEATURE_NAMES + ['label']].copy()
normalized_dataset = (raw_dataset[MODEL_FEATURE_NAMES] - baseline_mean) / baseline_std
normalized_dataset['label'] = raw_dataset['label']
baseline_statistics = pd.DataFrame({
    'feature': MODEL_FEATURE_NAMES,
    'healthy_mean': baseline_mean[MODEL_FEATURE_NAMES].to_numpy(),
    'healthy_std': baseline_std[MODEL_FEATURE_NAMES].to_numpy(),
})

ml_ready_dataset.to_csv('../data/fingerprint_dataset_ml_ready.csv', index=False)
normalized_dataset.to_csv('../data/fingerprint_dataset_normalized.csv', index=False)
baseline_statistics.to_csv('../data/healthy_baseline_statistics.csv', index=False)

print(f'Raw rows: {len(raw_dataset):,}; balanced classes: {raw_dataset.label.value_counts().iloc[0]:,} each')
print('Excluded constant features:', constant_features or 'None')
print('ML features:', MODEL_FEATURE_NAMES)
print('Saved ML-ready, normalized, and healthy-baseline CSV files to data/.')


Raw rows: 60,000; balanced classes: 12,000 each
Excluded constant features: None
ML features: ['rpm_drop_pct', 'settling_time_s', 'current_peak_a', 'current_rms_a', 'current_rise_pct', 'speed_rms_rad_s', 'overshoot_pct', 'speed_kurtosis', 'current_kurtosis', 'current_crest_factor']
Saved ML-ready, normalized, and healthy-baseline CSV files to data/.


In [ ]:
# Stage 6: mild-fault stress set. For TESTING only, never for training.
STRESS_SEED = 777            # different seed from training, so no sample overlap
STRESS_SAMPLES_PER_CLASS = 3_000

stress_raw, stress_audit = generate_fingerprint_dataset(
    samples_per_class=STRESS_SAMPLES_PER_CLASS, seed=STRESS_SEED, class_specs=STRESS_CLASS_SPECS
)
stress_raw['settling_time_s'] = calculate_extended_settling_time(stress_audit)

# Normalize with the ORIGINAL healthy baseline, not one recomputed from this set
baseline = pd.read_csv('../data/healthy_baseline_statistics.csv').set_index('feature')
mu = baseline.loc[MODEL_FEATURE_NAMES, 'healthy_mean']
sd = baseline.loc[MODEL_FEATURE_NAMES, 'healthy_std']
stress_norm = (stress_raw[MODEL_FEATURE_NAMES] - mu) / sd
stress_norm['label'] = stress_raw['label']

stress_norm.to_csv('../data/stress_dataset_normalized.csv', index=False)
stress_audit[['sample_id', 'label', 'fault_multiplier', 'severity']].to_csv(
    '../data/stress_metadata.csv', index=False)
print(stress_norm['label'].value_counts())
print('Saved stress_dataset_normalized.csv and stress_metadata.csv')